# Option 6: Economic Policy Uncertainty (EPU)
**Replication**: Baker, Bloom & Davis (2016) - Measuring Economic Policy Uncertainty
**Data**: EPU Index from local file + FRED macro data

**Key Question**: How does policy uncertainty affect investment and employment?

## 1. Setup and Data Download

In [ ]:
# Download data files if not already present
import os
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/JasmineHao/JasmineHao.github.io/main/econ6083/final-project/notebooks/data/"
DATA_FILES = ['us_epu_daily.csv', 'fred_indpro.csv', 'fred_unrate.csv']

os.makedirs('data', exist_ok=True)
for fname in DATA_FILES:
    if not os.path.exists(f'data/{fname}'):
        print(f"Downloading {fname} ...")
        urllib.request.urlretrieve(BASE_URL + fname, f'data/{fname}')
        print(f"  Saved to data/{fname}")
    else:
        print(f"Found local: data/{fname}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Download REAL EPU data from local file
# US Daily EPU Index
epu_df = pd.read_csv('data/us_epu_daily.csv')
print('Loaded EPU data from local file')
print(f'Shape: {epu_df.shape}')
print(f'Columns: {list(epu_df.columns)}')
print(epu_df.head(3))

## 2. Process EPU Data (Daily to Monthly)

In [ ]:
# EPU data has columns: day, month, year, daily_policy_index
# Convert to monthly average for merging with macro data

epu_df['date'] = pd.to_datetime(epu_df[['year', 'month', 'day']])
epu_df = epu_df.rename(columns={'daily_policy_index': 'epu'})

# Aggregate to monthly
epu_monthly = epu_df.set_index('date')['epu'].resample('ME').mean().reset_index()
epu_monthly['year_month'] = epu_monthly['date'].dt.to_period('M')

print(f"Monthly EPU shape: {epu_monthly.shape}")
print(f"Date range: {epu_monthly['date'].min()} to {epu_monthly['date'].max()}")
print(epu_monthly.head())

## 3. Load Macroeconomic Data

In [ ]:
# Load US macroeconomic data from local files

try:
    # Industrial Production (monthly)
    ip = pd.read_csv('data/fred_indpro.csv')
    ip.columns = ['date', 'industrial_production']
    ip['date'] = pd.to_datetime(ip['date'])
    ip['year_month'] = ip['date'].dt.to_period('M')
    
    # Unemployment Rate
    unemp = pd.read_csv('data/fred_unrate.csv')
    unemp.columns = ['date', 'unemployment']
    unemp['date'] = pd.to_datetime(unemp['date'])
    unemp['year_month'] = unemp['date'].dt.to_period('M')
    
    print("Successfully loaded macro data from local files!")
    macro_data = True
    
except Exception as e:
    print(f"Could not download FRED data: {e}")
    print("Using simulated macro data correlated with EPU...")
    
    # Create simulated correlated macro data
    dates = epu_monthly['date'].copy()
    n = len(dates)
    
    ip = pd.DataFrame({
        'year_month': epu_monthly['year_month'],
        'industrial_production': 100 - 0.05 * epu_monthly['epu'].values + np.random.normal(0, 2, n)
    })
    
    unemp = pd.DataFrame({
        'year_month': epu_monthly['year_month'],
        'unemployment': 5 + 0.01 * epu_monthly['epu'].values + np.random.normal(0, 0.5, n)
    })
    
    macro_data = False

print(f"\nIndustrial Production:")
print(ip.head(3))
print(f"\nUnemployment:")
print(unemp.head(3))

## 4. Merge and Explore Data

In [ ]:
# Merge all data by year_month
df = epu_monthly[['year_month', 'epu']].merge(
    ip[['year_month', 'industrial_production']], on='year_month', how='inner'
)
df = df.merge(
    unemp[['year_month', 'unemployment']], on='year_month', how='inner'
)

print(f"Merged dataset shape: {df.shape}")
print(f"\nDate range: {df['year_month'].min()} to {df['year_month'].max()}")
print(f"\nCorrelations with EPU:")
print(df[['epu', 'industrial_production', 'unemployment']].corr()['epu'].round(3))

df.head()

## 5. EPU Index Construction from Text

In [ ]:
# EPU keywords (Baker et al. 2016)
UNCERTAINTY_TERMS = ['uncertain', 'uncertainty', 'risk', 'volatile', 'unknown', 'unpredictable']
POLICY_TERMS = ['policy', 'regulation', 'legislation', 'federal reserve', 'government', 'congress']
ECONOMIC_TERMS = ['economy', 'growth', 'recession', 'inflation', 'unemployment', 'gdp']

def count_epu_articles(articles, unc_terms, pol_terms, eco_terms):
    """
    Count articles containing EPU keywords
    
    Article counts as EPU if it contains:
    - At least one uncertainty term AND
    - At least one policy term AND
    - At least one economic term
    
    Returns: EPU index (count per 100 articles)
    """
    epu_count = 0
    total = len(articles)
    
    for article in articles:
        text = article.lower()
        has_unc = any(term in text for term in unc_terms)
        has_pol = any(term in text for term in pol_terms)
        has_eco = any(term in text for term in eco_terms)
        
        if has_unc and has_pol and has_eco:
            epu_count += 1
    
    epu_index = (epu_count / total) * 100 if total > 0 else 0
    return epu_index

# Example: Simulate newspaper headlines
np.random.seed(42)
sample_articles = [
    "Federal Reserve uncertain about interest rate policy amid economic growth concerns",
    "Government announces new regulation for financial markets",
    "Stock market volatile as investors worry about recession risk",
    "Economy shows strong growth with low unemployment",
    "Congress debates tax policy amid budget uncertainty",
    "Federal Reserve maintains current policy stance",
    "Global markets face volatility due to trade policy uncertainty",
    "Economic indicators suggest continued expansion",
]

epu_idx = count_epu_articles(sample_articles, UNCERTAINTY_TERMS, POLICY_TERMS, ECONOMIC_TERMS)

print(f"Sample articles: {len(sample_articles)}")
print(f"EPU articles: {int(epu_idx * len(sample_articles) / 100)}")
print(f"EPU Index: {epu_idx:.1f} per 100 articles")

print("\nMatching articles:")
for article in sample_articles:
    text = article.lower()
    has_unc = any(term in text for term in UNCERTAINTY_TERMS)
    has_pol = any(term in text for term in POLICY_TERMS)
    has_eco = any(term in text for term in ECONOMIC_TERMS)
    match = "[EPU]" if (has_unc and has_pol and has_eco) else "[---]"
    print(f"  {match} {article}")

## 6. Regression Analysis

In [ ]:
# Analyze EPU effects on macro outcomes
outcomes = ['industrial_production', 'unemployment']
results = {}

for outcome in outcomes:
    # Contemporaneous
    corr = df['epu'].corr(df[outcome])
    
    # OLS: Outcome ~ EPU
    df_reg = df[['epu', outcome]].dropna()
    X = sm.add_constant(df_reg['epu'])
    model = sm.OLS(df_reg[outcome], X).fit()
    
    # OLS with lagged EPU (3 months)
    df['epu_lag3'] = df['epu'].shift(3)
    df_lag = df[['epu_lag3', outcome]].dropna()
    X_lag = sm.add_constant(df_lag['epu_lag3'])
    model_lag = sm.OLS(df_lag[outcome], X_lag).fit()
    
    results[outcome] = {
        'corr': corr,
        'coef': model.params[1],
        'se': model.bse[1],
        'r2': model.rsquared,
        'coef_lag3': model_lag.params[1],
        'r2_lag3': model_lag.rsquared
    }

results_df = pd.DataFrame(results).T

print("EPU Effects on Macroeconomic Outcomes:")
print("="*70)
print(results_df.round(4))

print("\nInterpretation:")
print("- corr: Contemporaneous correlation with EPU")
print("- coef: OLS coefficient (Outcome ~ EPU)")
print("- coef_lag3: OLS coefficient with 3-month lagged EPU")
print("- r2: Model fit")

## 7. Visualization

In [ ]:
# Convert year_month back to timestamp for plotting
df['date'] = df['year_month'].dt.to_timestamp()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: EPU time series
ax1 = axes[0, 0]
ax1.plot(df['date'], df['epu'], color='blue', linewidth=1)
ax1.set_ylabel('EPU Index')
ax1.set_title('Economic Policy Uncertainty Index')
ax1.grid(alpha=0.3)

# Plot 2: EPU vs Industrial Production
ax2 = axes[0, 1]
ax2.scatter(df['epu'], df['industrial_production'], alpha=0.3)
z = np.polyfit(df['epu'], df['industrial_production'], 1)
p = np.poly1d(z)
ax2.plot(df['epu'], p(df['epu']), "r--", alpha=0.8)
ax2.set_xlabel('EPU Index')
ax2.set_ylabel('Industrial Production')
ax2.set_title('EPU vs Industrial Production')
ax2.grid(alpha=0.3)

# Plot 3: EPU vs Unemployment
ax3 = axes[1, 0]
ax3.scatter(df['epu'], df['unemployment'], alpha=0.3, color='red')
z = np.polyfit(df['epu'], df['unemployment'], 1)
p = np.poly1d(z)
ax3.plot(df['epu'], p(df['epu']), "r--", alpha=0.8)
ax3.set_xlabel('EPU Index')
ax3.set_ylabel('Unemployment Rate (%)')
ax3.set_title('EPU vs Unemployment')
ax3.grid(alpha=0.3)

# Plot 4: Rolling correlation
ax4 = axes[1, 1]
rolling_corr = df['epu'].rolling(window=24).corr(df['unemployment'])
ax4.plot(df['date'], rolling_corr, color='purple')
ax4.set_xlabel('Year')
ax4.set_ylabel('Rolling Correlation (24-month)')
ax4.set_title('EPU-Unemployment Correlation Over Time')
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Interpreting Your Results

| Output | What it means | What to look for |
|---|---|---|
| **EPU index time series** | Monthly economic policy uncertainty | Spikes around elections, wars, financial crises |
| **Correlation with INDPRO** | Does EPU predict industrial production? | Negative correlation = uncertainty reduces output |
| **Regression coefficient** | Effect of EPU on macro outcomes | Statistically significant? Magnitude economically meaningful? |
| **Granger causality** | Does EPU lead or lag the economy? | If EPU Granger-causes output, it may be useful for forecasting |

**Key question**: Does economic policy uncertainty cause lower output, or does lower output cause more uncertainty? Your regression shows correlation; causal interpretation requires additional assumptions.

## Summary

This notebook replicates EPU index construction and analysis from Baker et al. (2016):

1. **EPU Data**: Real daily policy uncertainty index from local file, aggregated to monthly
2. **Text Matching**: Keyword-based article classification method
3. **Macro Effects**: EPU correlated with industrial production and unemployment
4. **Lagged Effects**: EPU effects materialize after 3-6 months

**Extensions to try**:
- Download China EPU data from data/china_epu.csv
- Build China-specific EPU using Chinese newspapers + Jieba
- Compare keyword-based vs BERT-based classification
- Separate indices for fiscal, monetary, trade policy
- Run VAR or local projections for dynamic effects